# 04 - CRF Baseline

**IT469 (King Saud University) - NER on Human-Written vs. LLM-Generated News**

## Purpose

This notebook trains the project's classical baseline: a **linear-chain Conditional Random Field (CRF)**
with hand-crafted features, as a reference point for how much the pre-trained transformer models
(`05_main_models.ipynb`) gain over a feature-based model that has no language pretraining at all. Both are
trained and evaluated on the same silver-labeled data from `03_silver_labeling.ipynb` (predictions of a
RoBERTa-large annotator fine-tuned on CoNLL-2003, validation F1 0.9699, test F1 0.9273), so this notebook
also establishes the shared evaluation code -- the 2x2 train/test design and the paired bootstrap -- that
notebook 05 will reuse without modification.

## What a CRF is, briefly

A linear-chain CRF [1] is a sequence model that predicts the most likely label for each token *given the
whole sequence*, using hand-designed features (the word itself, its shape, its neighbors) rather than
learned embeddings. Unlike a classifier that labels each token independently, it also learns which label
*transitions* are plausible (e.g. `I-PER` rarely follows `O`), using exact dynamic-programming inference.
It predates transformer-based NER by over a decade and needs no pretraining, making it a natural "how far
does feature engineering alone get you" baseline.

## Protocol (do not change, and do not add steps beyond these)

| Setting | Value | Source / reason |
|---|---|---|
| Model | `sklearn_crfsuite.CRF(algorithm="lbfgs", max_iterations=100, all_possible_transitions=True)` | Lafferty et al. [1]; CRFsuite [2] via sklearn-crfsuite [3]; settings from the sklearn-crfsuite CoNLL NER tutorial |
| Features (per token) | bias; lowercase word; suffixes (last 3/2 chars); prefixes (first 3/2 chars); is-upper; is-title; is-digit; same lowercase/is-title/is-upper for the previous and next word; BOS/EOS flags at sentence boundaries | Project proposal; sklearn-crfsuite tutorial feature set, without POS tags (our data has none) |
| Training data | Human side: `human_train_matched.jsonl` (token-count-matched to the LLM side). LLM side: `llm_train.jsonl` | Holding training text volume fixed isolates the effect of who wrote the text [6] |
| Hyperparameter grid | c1 (L1) in {0.1, 0.5, 1.0} x c2 (L2) in {0.01, 0.05, 0.1} -> 9 configs per side, 18 runs total | Centered on the sklearn-crfsuite tutorial's searched range; tuning the baseline keeps the comparison with the grid-searched transformers fair |
| Selection | Per side, the config with the highest validation micro F1 on that side's own validation set | Same rule as the transformer models; test sets never used for decisions |
| Evaluation | Each side's best model evaluated on BOTH `human_test` and `llm_test` -> 2x2 table. Entity-level micro P/R/F1, macro F1, per-type F1 via seqeval (default mode) | Project proposal; standard CoNLL evaluation |
| Confidence intervals | Paired bootstrap over TEST ARTICLES: 1,000 resamples with replacement, seed 42, 95% percentile intervals. One resample of article ids is applied consistently across all four comparisons, since human_test and llm_test contain the same articles | Project proposal; paired bootstrap resampling [7], [8]. Articles (not sentences) are resampled because sentences within an article are not independent |

## The 2x2 design and the four bootstrap comparisons

Two CRFs get trained (on human text, on LLM text) and each is evaluated on both test sets, giving four
(trained, tested) cells. The four paired-bootstrap comparisons test:

1. On **human test**: human-trained minus LLM-trained F1 -- does training on human text help on human text?
2. On **LLM test**: human-trained minus LLM-trained F1 -- does training source matter on LLM text?
3. **Human-trained** model: human test minus LLM test F1 -- does it transfer worse to LLM text?
4. **LLM-trained** model: human test minus LLM test F1 -- does it transfer worse to human text?

A difference is "reliable" if its 95% interval excludes zero.

## Limitation, stated plainly

Every score in this notebook measures agreement with the **silver labels** from notebook 03, not accuracy
against human-annotated ground truth on these specific news articles -- there is no such ground truth.
The annotator's own CoNLL-2003 test F1 (0.9273) is the only available proxy for how trustworthy those
silver labels are (see notebook 03's introduction for the full discussion).

## What gets saved, and where

Everything lives under the same shared Drive folder as notebooks 01-03: `ROOT = /content/drive/MyDrive/IT469-NER-Project`.

```
ROOT/
├── checkpoints/baseline/crf/{side}_best.joblib                      # best CRF per training side
├── results/baseline/crf/configs/{side}_c1{c1}_c2{c2}.json           # one per finished grid config
├── results/baseline/crf/grid_summary.csv
├── results/baseline/crf/best_configs.json
├── results/baseline/crf/crf_{trained}_on_{tested}.json              # 4 files, one per 2x2 cell
├── results/baseline/crf/predictions/crf_{trained}_on_{tested}.jsonl # 4 files: sent_id, article_id, predicted tags
├── results/baseline/crf/results_2x2.csv
├── results/baseline/crf/per_type_f1.csv
├── results/baseline/crf/bootstrap_ci.csv
├── results/baseline/crf/top_features.csv
├── results/baseline/crf/environment.json
├── results/baseline/crf/run_log.txt
└── results/_smoke/, checkpoints/_smoke/                             # smoke-test outputs only
```

## How to run

1. **No GPU needed** -- CPU runtime is fine. Edit **only** the Configuration cell (Section 1); leave
   `SMOKE_TEST = True` for a first run.
2. Runtime > Run all. The smoke test uses a small sample and a 2-config grid.
3. Once it prints "Smoke test passed", set `SMOKE_TEST = False` and Run all again for the real 18-config
   grid search plus evaluation. Expect roughly **2-5 minutes per config on CPU, about 1-1.5 hours total**,
   using around 3.5 GB of memory.
4. Reruns are safe: finished configs are skipped unless `FORCE_REBUILD = True`.

## References

In-text citations `[1]`-`[9]` point to the numbered list in Section 13 at the end of this notebook.


## 1. Configuration

This is the **only** cell you should edit.


In [ ]:
# Configuration: smoke/rebuild switches, seed, and the CRF hyperparameter grid.

SMOKE_TEST = False       # True = fast pipeline check on a small sample; set False for the real run
FORCE_REBUILD = False    # True = ignore existing outputs and recompute everything from scratch

SEED = 42
C1_GRID = [0.1, 0.5, 1.0]    # L1 regularization strength
C2_GRID = [0.01, 0.05, 0.1]  # L2 regularization strength
MAX_ITERATIONS = 100
N_BOOTSTRAP = 1000

SMOKE_N_TRAIN = 2000     # smoke test: training sentences per side
SMOKE_N_VAL = 500        # smoke test: validation sentences per side
SMOKE_N_TEST_ARTICLES = 60   # smoke test: test articles (both sides) for the 2x2 evaluation
SMOKE_C1_GRID = [0.1, 0.5]
SMOKE_C2_GRID = [0.05]
SMOKE_MAX_ITERATIONS = 20
SMOKE_N_BOOTSTRAP = 100

print("Active configuration")
print("---------------------")
print(f"SMOKE_TEST       : {SMOKE_TEST}")
print(f"FORCE_REBUILD     : {FORCE_REBUILD}")
print(f"SEED              : {SEED}")
print(f"C1_GRID           : {C1_GRID}{'  (smoke: ' + str(SMOKE_C1_GRID) + ')' if SMOKE_TEST else ''}")
print(f"C2_GRID           : {C2_GRID}{'  (smoke: ' + str(SMOKE_C2_GRID) + ')' if SMOKE_TEST else ''}")
print(f"MAX_ITERATIONS    : {MAX_ITERATIONS}{'  (smoke: ' + str(SMOKE_MAX_ITERATIONS) + ')' if SMOKE_TEST else ''}")
print(f"N_BOOTSTRAP       : {N_BOOTSTRAP}{'  (smoke: ' + str(SMOKE_N_BOOTSTRAP) + ')' if SMOKE_TEST else ''}")
print(f"Number of grid configs per side: {len(C1_GRID) * len(C2_GRID)}")


Active configuration
---------------------
SMOKE_TEST       : False
FORCE_REBUILD     : False
SEED              : 42
C1_GRID           : [0.1, 0.5, 1.0]
C2_GRID           : [0.01, 0.05, 0.1]
MAX_ITERATIONS    : 100
N_BOOTSTRAP       : 1000
Number of grid configs per side: 9


## 2. Setup

Install anything missing, import libraries, and record the environment. No GPU needed.


In [ ]:
!pip install -q -U pip setuptools wheel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 36.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.


In [ ]:
# Setup: install only what Colab is missing (do not pin versions -- avoid breaking Colab's preinstalled stack).
import importlib
import subprocess
import sys


def _ensure_installed(import_name, pip_name=None):
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])


_ensure_installed("sklearn_crfsuite", "sklearn-crfsuite")
_ensure_installed("seqeval")


In [ ]:
# Imports used throughout the notebook.
import gc
import importlib.metadata
import json
import logging
import platform
import time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import sklearn_crfsuite
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report


In [ ]:
# Record the environment (no GPU needed for a CRF).
ENV_INFO = {
    "date_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "sklearn_crfsuite_version": importlib.metadata.version("sklearn-crfsuite"),
    "python_crfsuite_version": importlib.metadata.version("python-crfsuite"),
    "scikit_learn_version": sklearn.__version__,
    "seqeval_version": importlib.metadata.version("seqeval"),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "smoke_test": SMOKE_TEST,
}
print(json.dumps(ENV_INFO, indent=2))


{
  "date_utc": "2026-10-16T13:08:32.764690+00:00",
  "python_version": "3.13.16",
  "sklearn_crfsuite_version": "0.5.0",
  "python_crfsuite_version": "0.9.12",
  "scikit_learn_version": "1.6.1",
  "seqeval_version": "1.2.2",
  "numpy_version": "2.1.3",
  "pandas_version": "2.2.3",
  "smoke_test": false
}


## 3. Mount Drive and paths

Mount Google Drive, resolve every path from `ROOT`, create folders, and set up logging that survives disconnects.


In [ ]:
# Mount Drive and locate the shared project folder (same ROOT as notebooks 01-03).
from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/IT469-NER-Project")
if not ROOT.exists():
    raise RuntimeError(
        f"{ROOT} does not exist. Add the shared folder as a shortcut in your own My Drive: "
        "Google Drive > Shared with me > IT469-NER-Project > (right-click) Organize > "
        "Add shortcut > My Drive. Then re-run this cell."
    )
print(f"ROOT resolved: {ROOT}")


Mounted at /content/drive
ROOT resolved: /content/drive/MyDrive/IT469-NER-Project


In [ ]:
# Resolve every other path from ROOT. Inputs (silver sentence files) are shared and real regardless of
# SMOKE_TEST; only this notebook's own outputs redirect to _smoke/ paths.
SILVER_DIR = ROOT / "data" / "silver"
SILVER_PATHS = {
    ("human", "train"): SILVER_DIR / "human_train_matched.jsonl",
    ("llm", "train"): SILVER_DIR / "llm_train.jsonl",
    ("human", "validation"): SILVER_DIR / "human_validation.jsonl",
    ("llm", "validation"): SILVER_DIR / "llm_validation.jsonl",
    ("human", "test"): SILVER_DIR / "human_test.jsonl",
    ("llm", "test"): SILVER_DIR / "llm_test.jsonl",
}

if SMOKE_TEST:
    CKPT_DIR = ROOT / "checkpoints" / "_smoke"
    RESULTS_DIR = ROOT / "results" / "_smoke"
else:
    CKPT_DIR = ROOT / "checkpoints" / "baseline" / "crf"
    RESULTS_DIR = ROOT / "results" / "baseline" / "crf"

CONFIGS_DIR = RESULTS_DIR / "configs"
PREDICTIONS_DIR = RESULTS_DIR / "predictions"
GRID_SUMMARY_CSV = RESULTS_DIR / "grid_summary.csv"
BEST_CONFIGS_JSON = RESULTS_DIR / "best_configs.json"
RESULTS_2X2_CSV = RESULTS_DIR / "results_2x2.csv"
PER_TYPE_F1_CSV = RESULTS_DIR / "per_type_f1.csv"
BOOTSTRAP_CI_CSV = RESULTS_DIR / "bootstrap_ci.csv"
TOP_FEATURES_CSV = RESULTS_DIR / "top_features.csv"
ENVIRONMENT_JSON = RESULTS_DIR / "environment.json"
RUN_LOG_TXT = RESULTS_DIR / "run_log.txt"

for d in [CKPT_DIR, CONFIGS_DIR, PREDICTIONS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"CKPT_DIR   : {CKPT_DIR}")
print(f"RESULTS_DIR: {RESULTS_DIR}")


CKPT_DIR   : /content/drive/MyDrive/IT469-NER-Project/checkpoints/baseline/crf
RESULTS_DIR: /content/drive/MyDrive/IT469-NER-Project/results/baseline/crf


In [ ]:
# Logger that writes to run_log.txt (appended, so it survives disconnects) AND prints to the cell output.
logger = logging.getLogger(f"crf_baseline_{'smoke' if SMOKE_TEST else 'full'}")
logger.setLevel(logging.INFO)
logger.handlers.clear()  # avoid duplicate handlers if this cell is re-run

_formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")

_file_handler = logging.FileHandler(RUN_LOG_TXT, mode="a", encoding="utf-8")
_file_handler.setFormatter(_formatter)
logger.addHandler(_file_handler)

_stream_handler = logging.StreamHandler()
_stream_handler.setFormatter(_formatter)
logger.addHandler(_stream_handler)


def save_json(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_jsonl(path):
    # Defined here (not Section 6) because Section 4 needs it to load the silver files; Section 6's
    # markdown note points back to this definition since notebook 05 reuses it unchanged too.
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


save_json(ENV_INFO, ENVIRONMENT_JSON)
logger.info(f"Session started. smoke_test={SMOKE_TEST} force_rebuild={FORCE_REBUILD} root={ROOT}")


2026-10-16 13:08:53,693 [INFO] Session started. smoke_test=False force_rebuild=False root=/content/drive/MyDrive/IT469-NER-Project
INFO:crf_baseline_full:Session started. smoke_test=False force_rebuild=False root=/content/drive/MyDrive/IT469-NER-Project


## 4. Load silver data + checks

Load the six silver files, verify tag well-formedness and the shared test-article requirement the paired bootstrap depends on.


In [ ]:
# Load all six silver files once.
VALID_TAGS = {"O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"}

silver = {key: load_jsonl(path) for key, path in SILVER_PATHS.items()}
for (side, split), records in silver.items():
    assert records, f"{side}_{split} is empty: {SILVER_PATHS[(side, split)]}"
    logger.info(f"Loaded {side}_{split}: {len(records)} sentences")


2026-10-16 13:09:05,469 [INFO] Loaded human_train: 55394 sentences
INFO:crf_baseline_full:Loaded human_train: 55394 sentences
2026-10-16 13:09:05,473 [INFO] Loaded llm_train: 50521 sentences
INFO:crf_baseline_full:Loaded llm_train: 50521 sentences
2026-10-16 13:09:05,475 [INFO] Loaded human_validation: 9060 sentences
INFO:crf_baseline_full:Loaded human_validation: 9060 sentences
2026-10-16 13:09:05,478 [INFO] Loaded llm_validation: 6221 sentences
INFO:crf_baseline_full:Loaded llm_validation: 6221 sentences
2026-10-16 13:09:05,479 [INFO] Loaded human_test: 9132 sentences
INFO:crf_baseline_full:Loaded human_test: 9132 sentences
2026-10-16 13:09:05,481 [INFO] Loaded llm_test: 6265 sentences
INFO:crf_baseline_full:Loaded llm_test: 6265 sentences


In [ ]:
# Assertions: tags/tokens align and are valid, and both test sets cover the SAME articles (the paired
# bootstrap resamples one shared list of article ids and applies it to both test sets).
for (side, split), records in silver.items():
    for rec in records:
        assert len(rec["tokens"]) == len(rec["ner_tags"]), (
            f"{side}_{split} {rec['sent_id']}: {len(rec['tokens'])} tokens but {len(rec['ner_tags'])} tags"
        )
        _bad_tags = set(rec["ner_tags"]) - VALID_TAGS
        assert not _bad_tags, f"{side}_{split} {rec['sent_id']}: invalid tag(s) {_bad_tags}"

human_test_articles = {r["article_id"] for r in silver[("human", "test")]}
llm_test_articles = {r["article_id"] for r in silver[("llm", "test")]}
assert human_test_articles == llm_test_articles, (
    f"human_test and llm_test must cover the same articles for the paired bootstrap -- "
    f"{len(human_test_articles - llm_test_articles)} article(s) only in human_test, "
    f"{len(llm_test_articles - human_test_articles)} article(s) only in llm_test"
)
logger.info(f"Checks OK: all tags/tokens aligned and valid; {len(human_test_articles)} shared test articles")


2026-10-16 13:09:05,707 [INFO] Checks OK: all tags/tokens aligned and valid; 290 shared test articles
INFO:crf_baseline_full:Checks OK: all tags/tokens aligned and valid; 290 shared test articles


In [ ]:
# Display sentences, tokens, and entities per file.
def _count_entities(records):
    return sum(1 for r in records for t in r["ner_tags"] if t.startswith("B-"))


_summary_rows = [
    {
        "side": side, "split": split, "n_sentences": len(records),
        "n_tokens": sum(len(r["tokens"]) for r in records),
        "n_entities": _count_entities(records),
    }
    for (side, split), records in silver.items()
]
display(pd.DataFrame(_summary_rows))


,side,split,n_sentences,n_tokens,n_entities
0,human,train,55394,1342507,91180
1,llm,train,50521,1342562,71947
2,human,validation,9060,219427,14530
3,llm,validation,6221,168119,8681
4,human,test,9132,217464,14526
5,llm,test,6265,167623,8878


In [ ]:
# Smoke test: shrink train/validation, and keep only the first 60 (sorted) shared test articles -- applied
# AFTER the real-scale checks and summary above, so those always reflect the true full dataset.
if SMOKE_TEST:
    for side in ["human", "llm"]:
        silver[(side, "train")] = silver[(side, "train")][:SMOKE_N_TRAIN]
        silver[(side, "validation")] = silver[(side, "validation")][:SMOKE_N_VAL]

    _smoke_test_articles = set(sorted(human_test_articles)[:SMOKE_N_TEST_ARTICLES])
    for side in ["human", "llm"]:
        silver[(side, "test")] = [r for r in silver[(side, "test")] if r["article_id"] in _smoke_test_articles]

    logger.info(
        f"SMOKE_TEST: train={len(silver[('human','train')])}/{len(silver[('llm','train')])} (human/llm), "
        f"val={len(silver[('human','validation')])}/{len(silver[('llm','validation')])}, "
        f"test_articles={len(_smoke_test_articles)}"
    )


## 5. Features

Per-token features exactly per the protocol table: the word's shape, affixes, and its immediate neighbors, plus sentence-boundary flags.


In [ ]:
# Per-token features: bias, word shape/affixes, and the same shape features for the previous/next word.
def token_features(tokens, i):
    word = tokens[i]
    features = {
        "bias": 1.0,
        "word.lower()": word.lower(),
        "word[-3:]": word[-3:],
        "word[-2:]": word[-2:],
        "word[:3]": word[:3],
        "word[:2]": word[:2],
        "word.isupper()": word.isupper(),
        "word.istitle()": word.istitle(),
        "word.isdigit()": word.isdigit(),
    }
    if i > 0:
        prev_word = tokens[i - 1]
        features.update({
            "-1:word.lower()": prev_word.lower(),
            "-1:word.istitle()": prev_word.istitle(),
            "-1:word.isupper()": prev_word.isupper(),
        })
    else:
        features["BOS"] = True
    if i < len(tokens) - 1:
        next_word = tokens[i + 1]
        features.update({
            "+1:word.lower()": next_word.lower(),
            "+1:word.istitle()": next_word.istitle(),
            "+1:word.isupper()": next_word.isupper(),
        })
    else:
        features["EOS"] = True
    return features


def sentence_features(tokens):
    return [token_features(tokens, i) for i in range(len(tokens))]


In [ ]:
# Show the features of one example token (the first word of the first human training sentence).
_example_tokens = silver[("human", "train")][0]["tokens"]
print(f"tokens: {_example_tokens}")
print(f"\nfeatures of token 0 ({_example_tokens[0]!r}):")
for k, v in token_features(_example_tokens, 0).items():
    print(f"  {k}: {v}")


tokens: ['(', 'See', 'The', 'Dead', 'Weather', "'s", 'video', 'for', 'Will', 'There', 'Be', 'Enough', 'Water', '?']

features of token 0 ('('):
  bias: 1.0
  word.lower(): (
  word[-3:]: (
  word[-2:]: (
  word[:3]: (
  word[:2]: (
  word.isupper(): False
  word.istitle(): False
  word.isdigit(): False
  BOS: True
  +1:word.lower(): see
  +1:word.istitle(): True
  +1:word.isupper(): False


## 6. Shared evaluation code

`load_jsonl` (Section 3), `extract_spans`, `evaluate`, `article_counts`, and `paired_bootstrap` are written here as clearly separated, reusable functions -- **notebook 05 reuses these exact functions unchanged** for the transformer models' 2x2 evaluation and bootstrap.


In [ ]:
# BIO span extraction, matching seqeval's default-mode interpretation exactly: a bare I-X not continuing a
# matching B-X/I-X starts a NEW entity of type X (the same rule used for BIO repair in notebook 03).
def extract_spans(tags):
    spans = []
    cur_type, cur_start = None, None
    for i, tag in enumerate(tags):
        if tag == "O":
            if cur_type is not None:
                spans.append((cur_type, cur_start, i))
                cur_type, cur_start = None, None
        elif tag.startswith("B-"):
            if cur_type is not None:
                spans.append((cur_type, cur_start, i))
            cur_type, cur_start = tag[2:], i
        elif tag.startswith("I-"):
            ent_type = tag[2:]
            if cur_type != ent_type:  # not continuing a matching entity -> starts a new one
                if cur_type is not None:
                    spans.append((cur_type, cur_start, i))
                cur_type, cur_start = ent_type, i
            # else: continues the current entity, nothing to do
    if cur_type is not None:
        spans.append((cur_type, cur_start, len(tags)))
    return spans


In [ ]:
# Standard seqeval evaluation: entity-level micro P/R/F1, macro F1, and the full per-type report.
def evaluate(gold_tag_lists, pred_tag_lists):
    report = classification_report(gold_tag_lists, pred_tag_lists, output_dict=True)
    return {
        "precision": precision_score(gold_tag_lists, pred_tag_lists),
        "recall": recall_score(gold_tag_lists, pred_tag_lists),
        "f1": f1_score(gold_tag_lists, pred_tag_lists),
        "macro_f1": report["macro avg"]["f1-score"],
        "report": report,
    }


In [ ]:
# Per-article (tp, fp, fn) counts (exact span-and-type match) -- precomputed once so the bootstrap can sum
# counts per resample instead of calling seqeval 1,000 times.
def article_counts(records, pred_tag_lists):
    counts = defaultdict(lambda: [0, 0, 0])
    for rec, pred_tags in zip(records, pred_tag_lists):
        gold_spans = set(extract_spans(rec["ner_tags"]))
        pred_spans = set(extract_spans(pred_tags))
        c = counts[rec["article_id"]]
        c[0] += len(gold_spans & pred_spans)   # tp
        c[1] += len(pred_spans - gold_spans)   # fp
        c[2] += len(gold_spans - pred_spans)   # fn
    return {aid: tuple(v) for aid, v in counts.items()}


In [ ]:
# Paired bootstrap over test articles: one resample of article ids is drawn per iteration and applied
# consistently to every (model, test_set) combination, so paired comparisons stay properly correlated.
def _micro_f1_from_counts(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0


def paired_bootstrap(counts_by_combo, comparisons, n_bootstrap, seed):
    '''counts_by_combo: {combo_key: {article_id: (tp, fp, fn)}}, all sharing the same article ids.
    comparisons: {name: (combo_key_a, combo_key_b)} -- each reported as a_f1 - b_f1.
    Returns a DataFrame: comparison, observed_difference, ci_low, ci_high, reliable.'''
    article_ids = sorted(next(iter(counts_by_combo.values())).keys())
    n = len(article_ids)
    tp_arr, fp_arr, fn_arr = {}, {}, {}
    for combo, by_article in counts_by_combo.items():
        tp_arr[combo] = np.array([by_article[aid][0] for aid in article_ids], dtype=np.int64)
        fp_arr[combo] = np.array([by_article[aid][1] for aid in article_ids], dtype=np.int64)
        fn_arr[combo] = np.array([by_article[aid][2] for aid in article_ids], dtype=np.int64)

    observed_f1 = {
        combo: _micro_f1_from_counts(tp_arr[combo].sum(), fp_arr[combo].sum(), fn_arr[combo].sum())
        for combo in counts_by_combo
    }

    rng = np.random.default_rng(seed)
    boot_diffs = {name: np.empty(n_bootstrap) for name in comparisons}
    for b in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)  # one resample of article positions, shared across all combos
        f1_this_draw = {
            combo: _micro_f1_from_counts(tp_arr[combo][idx].sum(), fp_arr[combo][idx].sum(), fn_arr[combo][idx].sum())
            for combo in counts_by_combo
        }
        for name, (a, c) in comparisons.items():
            boot_diffs[name][b] = f1_this_draw[a] - f1_this_draw[c]

    rows = []
    for name, (a, c) in comparisons.items():
        observed_diff = observed_f1[a] - observed_f1[c]
        ci_low, ci_high = np.percentile(boot_diffs[name], [2.5, 97.5])
        rows.append({
            "comparison": name, "observed_difference": observed_diff,
            "ci_low": ci_low, "ci_high": ci_high,
            "reliable": bool(ci_low > 0 or ci_high < 0),
        })
    return pd.DataFrame(rows)


In [ ]:
# Unit checks: extract_spans / article_counts must agree with real seqeval, including a bare I- after O.
_gold = ["O", "I-PER", "B-PER", "I-PER", "O", "B-ORG"]
_pred = ["O", "B-PER", "I-PER", "I-PER", "O", "B-MISC"]
# gold spans: I-PER(1,2) treated as its own entity (bare I- after O), then B-PER,I-PER -> (2,4); B-ORG -> (5,6)
assert extract_spans(_gold) == [("PER", 1, 2), ("PER", 2, 4), ("ORG", 5, 6)], extract_spans(_gold)
assert extract_spans(_pred) == [("PER", 1, 4), ("MISC", 5, 6)], extract_spans(_pred)

_seqeval_f1 = f1_score([_gold], [_pred])
_rec = {"article_id": 0, "ner_tags": _gold}
_counts = article_counts([_rec], [_pred])
_tp, _fp, _fn = _counts[0]
_our_f1 = _micro_f1_from_counts(_tp, _fp, _fn)
assert abs(_our_f1 - _seqeval_f1) < 1e-6, f"article_counts F1 {_our_f1} != seqeval F1 {_seqeval_f1}"

print("All unit checks passed:")
print(f'  extract_spans handles a bare "I-" after "O" as starting a new entity, matching seqeval')
print(f"  article_counts-derived F1 ({_our_f1:.4f}) matches seqeval's F1 ({_seqeval_f1:.4f}) exactly")


All unit checks passed:
  extract_spans handles a bare "I-" after "O" as starting a new entity, matching seqeval
  article_counts-derived F1 (0.0000) matches seqeval's F1 (0.0000) exactly


## 7. Grid search (resumable)

For each side, build features once, train all (c1, c2) configs (skipping finished ones), then select and save the best model by validation F1.


In [ ]:
# Active grid for this run (smoke uses a tiny 2-config grid).
active_c1_grid = SMOKE_C1_GRID if SMOKE_TEST else C1_GRID
active_c2_grid = SMOKE_C2_GRID if SMOKE_TEST else C2_GRID
active_max_iterations = SMOKE_MAX_ITERATIONS if SMOKE_TEST else MAX_ITERATIONS
active_n_bootstrap = SMOKE_N_BOOTSTRAP if SMOKE_TEST else N_BOOTSTRAP
grid_configs = [(c1, c2) for c1 in active_c1_grid for c2 in active_c2_grid]
print(f"Grid configs per side: {grid_configs}")


Grid configs per side: [(0.1, 0.01), (0.1, 0.05), (0.1, 0.1), (0.5, 0.01), (0.5, 0.05), (0.5, 0.1), (1.0, 0.01), (1.0, 0.05), (1.0, 0.1)]


In [ ]:
# Per-side resumable grid search: build features once, train/evaluate each config, then select the best.
best_configs = load_json(BEST_CONFIGS_JSON) if BEST_CONFIGS_JSON.exists() else {}

for side in ["human", "llm"]:
    best_model_path = CKPT_DIR / f"{side}_best.joblib"
    if best_model_path.exists() and not FORCE_REBUILD:
        logger.info(f"{side}: best model already exists, skipping grid search ({best_model_path})")
        continue

    logger.info(f"{side}: building features for {len(silver[(side, 'train')])} train / "
                f"{len(silver[(side, 'validation')])} validation sentences")
    X_train = [sentence_features(r["tokens"]) for r in silver[(side, "train")]]
    y_train = [r["ner_tags"] for r in silver[(side, "train")]]
    X_val = [sentence_features(r["tokens"]) for r in silver[(side, "validation")]]
    y_val = [r["ner_tags"] for r in silver[(side, "validation")]]

    trained_models = {}  # (c1, c2) -> crf object, only for configs freshly trained THIS run
    for c1, c2 in grid_configs:
        config_name = f"{side}_c1{c1}_c2{c2}"
        config_path = CONFIGS_DIR / f"{config_name}.json"
        if config_path.exists() and not FORCE_REBUILD:
            logger.info(f"{config_name}: skipped (already done)")
            continue

        crf = sklearn_crfsuite.CRF(
            algorithm="lbfgs", c1=c1, c2=c2, max_iterations=active_max_iterations,
            all_possible_transitions=True,
        )
        _start = time.time()
        crf.fit(X_train, y_train)
        train_time = time.time() - _start

        y_pred = crf.predict(X_val)
        metrics = evaluate(y_val, y_pred)
        config_result = {
            "side": side, "c1": c1, "c2": c2,
            "val_precision": metrics["precision"], "val_recall": metrics["recall"], "val_f1": metrics["f1"],
            "train_time_seconds": train_time,
        }
        save_json(config_result, config_path)
        trained_models[(c1, c2)] = crf
        logger.info(f"{config_name}: val_f1={metrics['f1']:.4f} (trained in {train_time:.1f}s)")

    # Select the best config for this side by validation F1 (reading every config's saved result, not
    # just the ones trained this run, so resumed runs select fairly across old and new configs alike).
    side_results = [load_json(CONFIGS_DIR / f"{side}_c1{c1}_c2{c2}.json") for c1, c2 in grid_configs]
    best = max(side_results, key=lambda r: r["val_f1"])
    best_c1, best_c2 = best["c1"], best["c2"]

    if (best_c1, best_c2) in trained_models:
        best_crf = trained_models[(best_c1, best_c2)]
    else:
        # The best config was already done in an earlier run, so its model object isn't in memory --
        # retrain it (cheap relative to the whole grid) purely to save the checkpoint.
        logger.info(f"{side}: retraining best config c1={best_c1} c2={best_c2} to save its model")
        best_crf = sklearn_crfsuite.CRF(
            algorithm="lbfgs", c1=best_c1, c2=best_c2, max_iterations=active_max_iterations,
            all_possible_transitions=True,
        )
        best_crf.fit(X_train, y_train)

    joblib.dump(best_crf, best_model_path)
    best_configs[side] = best
    save_json(best_configs, BEST_CONFIGS_JSON)
    logger.info(f"{side}: best config c1={best_c1} c2={best_c2} (val_f1={best['val_f1']:.4f}), saved {best_model_path}")

    del trained_models, best_crf, X_train, y_train, X_val, y_val
    gc.collect()


2026-10-16 13:09:06,162 [INFO] human: building features for 55394 train / 9060 validation sentences
INFO:crf_baseline_full:human: building features for 55394 train / 9060 validation sentences
2026-10-16 13:14:00,509 [INFO] human_c10.1_c20.01: val_f1=0.8540 (trained in 282.8s)
INFO:crf_baseline_full:human_c10.1_c20.01: val_f1=0.8540 (trained in 282.8s)
2026-10-16 13:19:47,132 [INFO] human_c10.1_c20.05: val_f1=0.8575 (trained in 339.2s)
INFO:crf_baseline_full:human_c10.1_c20.05: val_f1=0.8575 (trained in 339.2s)
2026-10-16 13:25:23,420 [INFO] human_c10.1_c20.1: val_f1=0.8588 (trained in 330.5s)
INFO:crf_baseline_full:human_c10.1_c20.1: val_f1=0.8588 (trained in 330.5s)
2026-10-16 13:31:00,676 [INFO] human_c10.5_c20.01: val_f1=0.8575 (trained in 329.3s)
INFO:crf_baseline_full:human_c10.5_c20.01: val_f1=0.8575 (trained in 329.3s)
2026-10-16 13:36:36,892 [INFO] human_c10.5_c20.05: val_f1=0.8571 (trained in 330.3s)
INFO:crf_baseline_full:human_c10.5_c20.05: val_f1=0.8571 (trained in 330.3s)


## 8. Grid summary

Collect every finished config into one table and show validation F1 as a c1 x c2 grid, per side.


In [ ]:
# Build grid_summary.csv from every finished configs/*.json file (both sides).
grid_rows = [load_json(p) for p in sorted(CONFIGS_DIR.glob("*.json"))]
grid_summary_df = pd.DataFrame(grid_rows).sort_values(["side", "val_f1"], ascending=[True, False]).reset_index(drop=True)
grid_summary_df.to_csv(GRID_SUMMARY_CSV, index=False)
display(grid_summary_df)


,side,c1,c2,val_precision,val_recall,val_f1,train_time_seconds
0,human,0.1,0.10,0.872947,0.845010,0.858752,330.495615
1,human,0.1,0.05,0.872286,0.843290,0.857543,339.207062
2,human,0.5,0.01,0.871332,0.844047,0.857472,329.313047
3,human,0.5,0.05,0.871797,0.842877,0.857093,330.326927
4,human,0.5,0.10,0.871108,0.839573,0.855050,330.917277
5,human,0.1,0.01,0.869147,0.839298,0.853962,282.757694
6,human,1.0,0.05,0.865287,0.834618,0.849676,328.122951
7,human,1.0,0.01,0.865120,0.833861,0.849203,315.423685
8,human,1.0,0.10,0.865589,0.833242,0.849108,336.166569
9,llm,0.1,0.10,0.891197,0.869946,0.880443,276.285402


In [ ]:
# Validation F1 as a c1 x c2 pivot table, per side.
best_configs = load_json(BEST_CONFIGS_JSON)
for side in ["human", "llm"]:
    side_df = grid_summary_df[grid_summary_df["side"] == side]
    pivot = side_df.pivot(index="c1", columns="c2", values="val_f1")
    print(f"\n{side} (best: c1={best_configs[side]['c1']}, c2={best_configs[side]['c2']}, val_f1={best_configs[side]['val_f1']:.4f}):")
    display(pivot)



human (best: c1=0.1, c2=0.1, val_f1=0.8588):


c2,0.01,0.05,0.10
c1,,,
0.1,0.853962,0.857543,0.858752
0.5,0.857472,0.857093,0.855050
1.0,0.849203,0.849676,0.849108



llm (best: c1=0.1, c2=0.1, val_f1=0.8804):


c2,0.01,0.05,0.10
c1,,,
0.1,0.877301,0.880102,0.880443
0.5,0.876760,0.877126,0.876566
1.0,0.871914,0.872611,0.871314


## 9. 2x2 evaluation

Each side's best model, tested on both test sets: train source x test source.


In [ ]:
# Load both best models, predict on both test sets, save per-cell results and predictions.
pred_by_combo = {}  # (trained, tested) -> list of predicted tag lists, aligned with silver[(tested,'test')]
results_2x2_rows, per_type_rows = [], []

for trained in ["human", "llm"]:
    crf = joblib.load(CKPT_DIR / f"{trained}_best.joblib")
    for tested in ["human", "llm"]:
        test_records = silver[(tested, "test")]
        X_test = [sentence_features(r["tokens"]) for r in test_records]
        y_test = [r["ner_tags"] for r in test_records]
        y_pred = crf.predict(X_test)
        pred_by_combo[(trained, tested)] = y_pred

        metrics = evaluate(y_test, y_pred)
        cell_result = {
            "trained_on": trained, "tested_on": tested,
            "precision": metrics["precision"], "recall": metrics["recall"], "f1": metrics["f1"],
            "macro_f1": metrics["macro_f1"], "report": metrics["report"],
        }
        save_json(cell_result, RESULTS_DIR / f"crf_{trained}_on_{tested}.json")

        pred_records = [
            {"sent_id": r["sent_id"], "article_id": r["article_id"], "predicted_tags": tags}
            for r, tags in zip(test_records, y_pred)
        ]
        with open(PREDICTIONS_DIR / f"crf_{trained}_on_{tested}.jsonl", "w", encoding="utf-8") as f:
            for rec in pred_records:
                f.write(json.dumps(rec) + "\n")

        results_2x2_rows.append({
            "trained_on": trained, "tested_on": tested,
            "micro_f1": metrics["f1"], "macro_f1": metrics["macro_f1"],
            "precision": metrics["precision"], "recall": metrics["recall"],
        })
        for etype, type_metrics in metrics["report"].items():
            if etype in ("micro avg", "macro avg", "weighted avg"):
                continue
            per_type_rows.append({
                "trained_on": trained, "tested_on": tested, "entity_type": etype,
                "f1": type_metrics["f1-score"], "precision": type_metrics["precision"],
                "recall": type_metrics["recall"], "support": type_metrics["support"],
            })
        logger.info(f"trained={trained} tested={tested}: micro_f1={metrics['f1']:.4f} macro_f1={metrics['macro_f1']:.4f}")

    del crf
    gc.collect()

results_2x2_df = pd.DataFrame(results_2x2_rows)
results_2x2_df.to_csv(RESULTS_2X2_CSV, index=False)
per_type_f1_df = pd.DataFrame(per_type_rows)
per_type_f1_df.to_csv(PER_TYPE_F1_CSV, index=False)

display(results_2x2_df)
display(per_type_f1_df)


2026-10-16 14:42:06,333 [INFO] trained=human tested=human: micro_f1=0.8530 macro_f1=0.8441
INFO:crf_baseline_full:trained=human tested=human: micro_f1=0.8530 macro_f1=0.8441
2026-10-16 14:42:17,136 [INFO] trained=human tested=llm: micro_f1=0.8691 macro_f1=0.8629
INFO:crf_baseline_full:trained=human tested=llm: micro_f1=0.8691 macro_f1=0.8629
2026-10-16 14:42:25,453 [INFO] trained=llm tested=human: micro_f1=0.8049 macro_f1=0.7959
INFO:crf_baseline_full:trained=llm tested=human: micro_f1=0.8049 macro_f1=0.7959
2026-10-16 14:42:32,366 [INFO] trained=llm tested=llm: micro_f1=0.8714 macro_f1=0.8650
INFO:crf_baseline_full:trained=llm tested=llm: micro_f1=0.8714 macro_f1=0.8650


,trained_on,tested_on,micro_f1,macro_f1,precision,recall
0,human,human,0.853015,0.844144,0.864968,0.841388
1,human,llm,0.869110,0.862938,0.878307,0.860104
2,llm,human,0.804864,0.795936,0.814770,0.795195
3,llm,llm,0.871382,0.865037,0.883436,0.859653


,trained_on,tested_on,entity_type,f1,precision,recall,support
0,human,human,LOC,0.909589,0.915722,0.903538,3307
1,human,human,MISC,0.778867,0.800189,0.758652,3352
2,human,human,ORG,0.780538,0.805843,0.756775,2989
3,human,human,PER,0.907582,0.907210,0.907954,4878
4,human,llm,LOC,0.924366,0.927604,0.921150,2156
5,human,llm,MISC,0.830926,0.857798,0.805687,2321
6,human,llm,ORG,0.786380,0.789136,0.783643,1724
7,human,llm,PER,0.910079,0.912815,0.907359,2677
8,llm,human,LOC,0.866234,0.885380,0.847898,3307
9,llm,human,MISC,0.738645,0.737437,0.739857,3352


## 10. Paired bootstrap

The four comparisons from Section 0, with 95% percentile intervals over resampled test articles.


In [ ]:
# Precompute per-article (tp, fp, fn) for all 4 (trained, tested) combinations.
counts_by_combo = {
    (trained, tested): article_counts(silver[(tested, "test")], pred_by_combo[(trained, tested)])
    for trained in ["human", "llm"] for tested in ["human", "llm"]
}


In [ ]:
# Verify the summed per-article counts reproduce seqeval's micro F1 EXACTLY before trusting the bootstrap.
for (trained, tested), by_article in counts_by_combo.items():
    tp = sum(c[0] for c in by_article.values())
    fp = sum(c[1] for c in by_article.values())
    fn = sum(c[2] for c in by_article.values())
    our_f1 = _micro_f1_from_counts(tp, fp, fn)
    seqeval_f1 = f1_score(
        [r["ner_tags"] for r in silver[(tested, "test")]], pred_by_combo[(trained, tested)]
    )
    assert abs(our_f1 - seqeval_f1) < 1e-6, (
        f"trained={trained} tested={tested}: article_counts F1 ({our_f1:.6f}) != seqeval F1 ({seqeval_f1:.6f})"
    )
logger.info("Verified: summed per-article counts reproduce seqeval's micro F1 exactly for all 4 combinations")


2026-10-16 14:42:35,000 [INFO] Verified: summed per-article counts reproduce seqeval's micro F1 exactly for all 4 combinations
INFO:crf_baseline_full:Verified: summed per-article counts reproduce seqeval's micro F1 exactly for all 4 combinations


In [ ]:
# The four comparisons from the protocol: (a_combo) minus (b_combo).
comparisons = {
    "human_test: human_trained - llm_trained": (("human", "human"), ("llm", "human")),
    "llm_test: human_trained - llm_trained": (("human", "llm"), ("llm", "llm")),
    "human_trained: human_test - llm_test": (("human", "human"), ("human", "llm")),
    "llm_trained: human_test - llm_test": (("llm", "human"), ("llm", "llm")),
}
bootstrap_ci_df = paired_bootstrap(counts_by_combo, comparisons, active_n_bootstrap, SEED)
bootstrap_ci_df.to_csv(BOOTSTRAP_CI_CSV, index=False)

_display_df = bootstrap_ci_df.copy()
_display_df["reliable"] = _display_df["reliable"].map({True: "reliable (CI excludes 0)", False: "not reliable"})
display(_display_df)


,comparison,observed_difference,ci_low,ci_high,reliable
0,human_test: human_trained - llm_trained,0.048151,0.040786,0.055326,reliable (CI excludes 0)
1,llm_test: human_trained - llm_trained,-0.002272,-0.010376,0.005193,not reliable
2,human_trained: human_test - llm_test,-0.016095,-0.027899,-0.004247,reliable (CI excludes 0)
3,llm_trained: human_test - llm_test,-0.066518,-0.079658,-0.053039,reliable (CI excludes 0)


## 11. Feature weights

For each best model, the top 20 positive state features per entity type -- a window into what the CRF actually learned.


In [ ]:
# Top 20 positive state features per entity type per side, from crf.state_features_: {(feature, label): weight}.
ENTITY_TYPES = ["PER", "ORG", "LOC", "MISC"]
top_feature_rows = []
for side in ["human", "llm"]:
    crf = joblib.load(CKPT_DIR / f"{side}_best.joblib")
    for etype in ENTITY_TYPES:
        _labels = {f"B-{etype}", f"I-{etype}"}
        _candidates = [
            (feature, label, weight)
            for (feature, label), weight in crf.state_features_.items()
            if label in _labels and weight > 0
        ]
        _candidates.sort(key=lambda x: -x[2])
        for rank, (feature, label, weight) in enumerate(_candidates[:20], start=1):
            top_feature_rows.append({
                "side": side, "entity_type": etype, "rank": rank,
                "feature": feature, "label": label, "weight": weight,
            })
    del crf
    gc.collect()

top_features_df = pd.DataFrame(top_feature_rows)
top_features_df.to_csv(TOP_FEATURES_CSV, index=False)


In [ ]:
# Show one entity type per side as an example (PER -- usually the most interpretable).
for side in ["human", "llm"]:
    print(f"\nTop PER features ({side}-trained model):")
    display(top_features_df[(top_features_df["side"] == side) & (top_features_df["entity_type"] == "PER")].head(20))



Top PER features (human-trained model):


,side,entity_type,rank,feature,label,weight
0,human,PER,1,word[:2]:Mc,B-PER,6.581067
1,human,PER,2,word[:2]:Mc,I-PER,5.653967
2,human,PER,3,word.lower():evers,B-PER,5.056958
3,human,PER,4,word.lower():brian,B-PER,4.349134
4,human,PER,5,word.lower():wright,B-PER,4.325869
5,human,PER,6,word.lower():trump,B-PER,4.324733
6,human,PER,7,word.lower():questlove,B-PER,4.313820
7,human,PER,8,-1:word.lower():$,I-PER,4.293576
8,human,PER,9,word.lower():underwood,B-PER,4.271570
9,human,PER,10,word.lower():thaksin,B-PER,4.238724



Top PER features (llm-trained model):


,side,entity_type,rank,feature,label,weight
80,llm,PER,1,word[:2]:Mc,B-PER,5.542054
81,llm,PER,2,word[:2]:Mc,I-PER,5.126413
82,llm,PER,3,-1:word.lower():mayor,B-PER,4.672727
83,llm,PER,4,-1:word.lower():judge,B-PER,4.374900
84,llm,PER,5,word.lower():borland,B-PER,4.186485
85,llm,PER,6,-1:word.lower():duvet,B-PER,4.182755
86,llm,PER,7,-1:word.lower():soul,B-PER,4.167617
87,llm,PER,8,-1:word.lower():president,B-PER,4.134256
88,llm,PER,9,-1:word.lower():dr.,B-PER,4.081752
89,llm,PER,10,+1:word.lower():jr.,I-PER,4.076073


## 12. Summary


In [ ]:
# One short, final overview: best config per side, the 2x2 micro F1 table, and the bootstrap differences.
print("Best config per side:")
for side in ["human", "llm"]:
    b = best_configs[side]
    print(f"  {side}: c1={b['c1']}, c2={b['c2']}, val_f1={b['val_f1']:.4f}")

print("\n2x2 micro F1 (rows=trained_on, cols=tested_on):")
display(results_2x2_df.pivot(index="trained_on", columns="tested_on", values="micro_f1"))

print("\nPaired bootstrap differences (95% CI over test articles):")
display(bootstrap_ci_df)

if SMOKE_TEST:
    print("\nSmoke test passed. Set SMOKE_TEST = False and run all cells.")


Best config per side:
  human: c1=0.1, c2=0.1, val_f1=0.8588
  llm: c1=0.1, c2=0.1, val_f1=0.8804

2x2 micro F1 (rows=trained_on, cols=tested_on):


tested_on,human,llm
trained_on,,
human,0.853015,0.869110
llm,0.804864,0.871382



Paired bootstrap differences (95% CI over test articles):


,comparison,observed_difference,ci_low,ci_high,reliable
0,human_test: human_trained - llm_trained,0.048151,0.040786,0.055326,True
1,llm_test: human_trained - llm_trained,-0.002272,-0.010376,0.005193,False
2,human_trained: human_test - llm_test,-0.016095,-0.027899,-0.004247,True
3,llm_trained: human_test - llm_test,-0.066518,-0.079658,-0.053039,True


## 13. Leakage and overfitting checks

Added after reviewing the first real run, because the cross-domain F1 gaps (largest: -6.7 points,
LLM-trained model tested on human text) are large enough to deserve direct evidence, not just trust in
notebook 01's upstream split logic. Two checks:

1. **Independent re-verification that train/validation/test share no article ids.** Notebook 01 already
   asserts this at the source (grouped, topic-stratified split), but re-checking it here, on the exact
   data this notebook trained and evaluated on, is cheap insurance against silent upstream data issues
   (this project has already had Drive files go missing once -- see `CLAUDE.md`).
2. **Entity-vocabulary coverage between each train set and each test set.** The CRF's features
   (`word.lower()`, prefixes/suffixes) make it substantially a lexical lookup table for proper nouns, so
   its cross-domain performance should track how much entity vocabulary the test set shares with the
   training set, not just topic or style differences. This also connects directly to notebook 03's entity
   preservation finding (~44% of a journalist's entity strings survive into GPT-3.5's rewrite of the same
   story) -- if that's really the mechanism, the vocabulary-coverage pattern below should line up with the
   F1 2x2 pattern from Section 9.


In [ ]:
# Check 1: train/validation/test must share NO article ids (re-verified independently of notebook 01).
_split_articles = {}
for split in ["train", "validation", "test"]:
    _split_articles[split] = {r["article_id"] for r in silver[("human", split)]} | {r["article_id"] for r in silver[("llm", split)]}

_overlap_train_val = _split_articles["train"] & _split_articles["validation"]
_overlap_train_test = _split_articles["train"] & _split_articles["test"]
_overlap_val_test = _split_articles["validation"] & _split_articles["test"]

assert not _overlap_train_val, f"{len(_overlap_train_val)} article(s) in both train and validation: {sorted(_overlap_train_val)[:10]}"
assert not _overlap_train_test, f"{len(_overlap_train_test)} article(s) in both train and test: {sorted(_overlap_train_test)[:10]}"
assert not _overlap_val_test, f"{len(_overlap_val_test)} article(s) in both validation and test: {sorted(_overlap_val_test)[:10]}"

logger.info(
    f"Leakage check 1 OK: 0 shared article ids across train ({len(_split_articles['train'])} articles) / "
    f"validation ({len(_split_articles['validation'])}) / test ({len(_split_articles['test'])})"
)
print("No article-id overlap between train, validation, and test.")

2026-10-16 15:12:08,785 [INFO] Leakage check 1 OK: 0 shared article ids across train (2315 articles) / validation (286) / test (290)
INFO:crf_baseline_full:Leakage check 1 OK: 0 shared article ids across train (2315 articles) / validation (286) / test (290)


No article-id overlap between train, validation, and test.


In [ ]:
# Check 2: entity-vocabulary coverage -- for each (train_side, test_side) pair, what fraction of the test
# set's entity MENTIONS (lowercased string) also appear in that training set's entity vocabulary.
def entity_vocab(records):
    vocab = set()
    for r in records:
        for (etype, s, e) in extract_spans(r["ner_tags"]):
            vocab.add(" ".join(r["tokens"][s:e]).lower())
    return vocab


train_vocab = {side: entity_vocab(silver[(side, "train")]) for side in ["human", "llm"]}

coverage_rows = []
for test_side in ["human", "llm"]:
    _occurrences = []
    for r in silver[(test_side, "test")]:
        for (etype, s, e) in extract_spans(r["ner_tags"]):
            _occurrences.append(" ".join(r["tokens"][s:e]).lower())
    n_total = len(_occurrences)
    for train_side in ["human", "llm"]:
        n_covered = sum(1 for text in _occurrences if text in train_vocab[train_side])
        coverage_rows.append({
            "train_side": train_side, "test_side": test_side,
            "n_test_entity_mentions": n_total,
            "pct_covered_by_train_vocab": round(n_covered / n_total * 100, 2) if n_total else 0.0,
        })

coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv(RESULTS_DIR / "entity_vocab_coverage.csv", index=False)
logger.info(f"Saved {RESULTS_DIR / 'entity_vocab_coverage.csv'}")

print("Entity-vocabulary coverage (% of test entity mentions whose exact string appears in the training set):")
display(coverage_df.pivot(index="train_side", columns="test_side", values="pct_covered_by_train_vocab"))

print("\nFor comparison, the micro-F1 2x2 table from Section 9:")
display(results_2x2_df.pivot(index="trained_on", columns="tested_on", values="micro_f1"))

2026-10-16 15:12:37,839 [INFO] Saved /content/drive/MyDrive/IT469-NER-Project/results/baseline/crf/entity_vocab_coverage.csv
INFO:crf_baseline_full:Saved /content/drive/MyDrive/IT469-NER-Project/results/baseline/crf/entity_vocab_coverage.csv


Entity-vocabulary coverage (% of test entity mentions whose exact string appears in the training set):


test_side,human,llm
train_side,,
human,71.99,77.63
llm,65.76,74.56



For comparison, the micro-F1 2x2 table from Section 9:


tested_on,human,llm
trained_on,,
human,0.853015,0.869110
llm,0.804864,0.871382


## 14. RQ3 -- entity types and topics most affected by the origin of the text

Answers the project's third research question using results already saved to Drive from Section 9/11 --
no retraining, no feature rebuilding, **only Drive needs to be (re)mounted** (useful after a runtime
disconnect, since the grid search is the expensive part and doesn't need to run again).

"Origin of the text" = whether the TEST text is human-written or LLM-generated, holding the trained model
fixed -- i.e. the same comparison as bootstrap rows 3 and 4 in Section 10 (human test vs. llm test, for
each trained model), broken down by entity type (reusing `per_type_f1.csv`) and by topic (recomputed here
from the saved predictions + the original test files, joined on `sent_id`).

**Caveat:** unlike the headline Section 10 numbers, the breakdowns below are point estimates with no
per-cell bootstrap interval -- splitting ~9,132/6,265 test sentences across 4-6 topics or 4 entity types
leaves each individual cell too small for a reliable per-cell CI. Use these to see WHERE the
already-validated overall effect concentrates, not as newly significance-tested findings on their own.


In [1]:
# Self-contained setup: only remounts Drive and redefines the two small helpers needed below -- does NOT
# depend on any earlier cell's in-memory state, so this works right after a fresh reconnect.
from pathlib import Path
from collections import defaultdict
import json

from google.colab import drive
import pandas as pd

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/IT469-NER-Project")
SILVER_DIR = ROOT / "data" / "silver"
RESULTS_DIR = ROOT / "results" / "baseline" / "crf"          # the real run's results, not _smoke
PREDICTIONS_DIR = RESULTS_DIR / "predictions"
assert RESULTS_DIR.exists(), f"{RESULTS_DIR} not found -- has the real (non-smoke) run finished Section 9?"


def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def extract_spans(tags):
    # Same BIO span logic as Section 6 -- copied here so this cell has no dependency on earlier cells.
    spans = []
    cur_type, cur_start = None, None
    for i, tag in enumerate(tags):
        if tag == "O":
            if cur_type is not None:
                spans.append((cur_type, cur_start, i))
                cur_type, cur_start = None, None
        elif tag.startswith("B-"):
            if cur_type is not None:
                spans.append((cur_type, cur_start, i))
            cur_type, cur_start = tag[2:], i
        elif tag.startswith("I-"):
            ent_type = tag[2:]
            if cur_type != ent_type:
                if cur_type is not None:
                    spans.append((cur_type, cur_start, i))
                cur_type, cur_start = ent_type, i
    if cur_type is not None:
        spans.append((cur_type, cur_start, len(tags)))
    return spans


def micro_f1_from_counts(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0


Mounted at /content/drive


In [2]:
# Per-topic F1: join each (trained, tested) combo's saved predictions onto the original test file (which
# has `topic`), via sent_id, and accumulate span-level (tp, fp, fn) per topic.
gold_by_tested = {
    side: {r["sent_id"]: r for r in load_jsonl(SILVER_DIR / f"{side}_test.jsonl")}
    for side in ["human", "llm"]
}

topic_counts = defaultdict(lambda: [0, 0, 0])  # (trained, tested, topic) -> [tp, fp, fn]
for trained in ["human", "llm"]:
    for tested in ["human", "llm"]:
        pred_path = PREDICTIONS_DIR / f"crf_{trained}_on_{tested}.jsonl"
        assert pred_path.exists(), f"Missing {pred_path} -- Section 9 must have finished for this combo"
        gold_lookup = gold_by_tested[tested]
        for pred_rec in load_jsonl(pred_path):
            gold_rec = gold_lookup.get(pred_rec["sent_id"])
            assert gold_rec is not None, f"{pred_rec['sent_id']} not found in {tested}_test.jsonl"
            gold_spans = set(extract_spans(gold_rec["ner_tags"]))
            pred_spans = set(extract_spans(pred_rec["predicted_tags"]))
            key = (trained, tested, gold_rec["topic"])
            topic_counts[key][0] += len(gold_spans & pred_spans)   # tp
            topic_counts[key][1] += len(pred_spans - gold_spans)   # fp
            topic_counts[key][2] += len(gold_spans - pred_spans)   # fn

topic_f1_rows = [
    {"trained_on": trained, "tested_on": tested, "topic": topic,
     "f1": micro_f1_from_counts(*counts), "n_entities_gold": counts[0] + counts[2]}
    for (trained, tested, topic), counts in topic_counts.items()
]
topic_f1_df = pd.DataFrame(topic_f1_rows)
topic_f1_df.to_csv(RESULTS_DIR / "per_topic_f1.csv", index=False)
display(topic_f1_df.sort_values(["trained_on", "topic", "tested_on"]))


,trained_on,tested_on,topic,f1,n_entities_gold
3,human,human,Celebrity,0.750776,2961
9,human,llm,Celebrity,0.800000,1765
0,human,human,History and religion,0.875838,3405
6,human,llm,History and religion,0.855783,2104
4,human,human,Politics and government,0.898001,1997
10,human,llm,Politics and government,0.924635,1247
1,human,human,Science and technology,0.888042,1726
7,human,llm,Science and technology,0.891440,1059
2,human,human,Social culture and civil rights,0.877247,2301
8,human,llm,Social culture and civil rights,0.887334,1427


In [3]:
# Rank topics by |F1 gap| between human test and llm test, for each trained model -- this is the
# "which topics are most affected by text origin" answer.
topic_pivot = topic_f1_df.pivot_table(index=["trained_on", "topic"], columns="tested_on", values="f1").reset_index()
topic_pivot["origin_gap"] = topic_pivot["human"] - topic_pivot["llm"]
topic_pivot["abs_gap"] = topic_pivot["origin_gap"].abs()
topic_pivot = topic_pivot.sort_values(["trained_on", "abs_gap"], ascending=[True, False])
topic_pivot.to_csv(RESULTS_DIR / "per_topic_origin_gap.csv", index=False)

print("Topics ranked by |F1(human test) - F1(llm test)|, per trained model:")
display(topic_pivot[["trained_on", "topic", "human", "llm", "origin_gap"]])


Topics ranked by |F1(human test) - F1(llm test)|, per trained model:


tested_on,trained_on,topic,human,llm,origin_gap
0,human,Celebrity,0.750776,0.800000,-0.049224
5,human,Sports,0.860664,0.891795,-0.031132
2,human,Politics and government,0.898001,0.924635,-0.026635
1,human,History and religion,0.875838,0.855783,0.020055
4,human,Social culture and civil rights,0.877247,0.887334,-0.010087
3,human,Science and technology,0.888042,0.891440,-0.003397
11,llm,Sports,0.808019,0.901945,-0.093926
6,llm,Celebrity,0.694813,0.787755,-0.092942
10,llm,Social culture and civil rights,0.819335,0.890282,-0.070947
8,llm,Politics and government,0.870183,0.922389,-0.052207


In [4]:
# Entity types ranked by |F1 gap| between human test and llm test, per trained model -- reuses the
# per_type_f1.csv already saved in Section 9, no recomputation needed.
per_type_f1_df = pd.read_csv(RESULTS_DIR / "per_type_f1.csv")
type_pivot = per_type_f1_df.pivot_table(index=["trained_on", "entity_type"], columns="tested_on", values="f1").reset_index()
type_pivot["origin_gap"] = type_pivot["human"] - type_pivot["llm"]
type_pivot["abs_gap"] = type_pivot["origin_gap"].abs()
type_pivot = type_pivot.sort_values(["trained_on", "abs_gap"], ascending=[True, False])
type_pivot.to_csv(RESULTS_DIR / "per_type_origin_gap.csv", index=False)

print("Entity types ranked by |F1(human test) - F1(llm test)|, per trained model:")
display(type_pivot[["trained_on", "entity_type", "human", "llm", "origin_gap"]])


Entity types ranked by |F1(human test) - F1(llm test)|, per trained model:


tested_on,trained_on,entity_type,human,llm,origin_gap
1,human,MISC,0.778867,0.830926,-0.052060
0,human,LOC,0.909589,0.924366,-0.014777
2,human,ORG,0.780538,0.786380,-0.005841
3,human,PER,0.907582,0.910079,-0.002497
5,llm,MISC,0.738645,0.834725,-0.096080
6,llm,ORG,0.714604,0.791805,-0.077201
4,llm,LOC,0.866234,0.923545,-0.057310
7,llm,PER,0.864261,0.910073,-0.045811


In [5]:
# RQ3 answer: entity type / topic combinations where origin (human vs LLM test text) matters most,
# for the model where the overall origin effect was largest and bootstrap-reliable (llm-trained, Section 10).
print("RQ3 -- most affected by text origin (llm-trained model, where the overall effect was largest and reliable):")
print("\nBy entity type:")
display(type_pivot[type_pivot["trained_on"] == "llm"][["entity_type", "human", "llm", "origin_gap"]])
print("\nBy topic:")
display(topic_pivot[topic_pivot["trained_on"] == "llm"][["topic", "human", "llm", "origin_gap"]])


RQ3 -- most affected by text origin (llm-trained model, where the overall effect was largest and reliable):

By entity type:


tested_on,entity_type,human,llm,origin_gap
5,MISC,0.738645,0.834725,-0.096080
6,ORG,0.714604,0.791805,-0.077201
4,LOC,0.866234,0.923545,-0.057310
7,PER,0.864261,0.910073,-0.045811



By topic:


tested_on,topic,human,llm,origin_gap
11,Sports,0.808019,0.901945,-0.093926
6,Celebrity,0.694813,0.787755,-0.092942
10,Social culture and civil rights,0.819335,0.890282,-0.070947
8,Politics and government,0.870183,0.922389,-0.052207
7,History and religion,0.821100,0.869398,-0.048298
9,Science and technology,0.861160,0.889741,-0.028581


## 15. References

[1] J. Lafferty, A. McCallum, and F. Pereira, "Conditional random fields: Probabilistic models for
segmenting and labeling sequence data," in Proc. 18th Int. Conf. Machine Learning (ICML), 2001,
pp. 282-289.

[2] N. Okazaki, "CRFsuite: A fast implementation of conditional random fields (CRFs)," 2007.
https://www.chokkan.org/software/crfsuite/

[3] M. Korobov et al., "sklearn-crfsuite," https://github.com/TeamHG-Memex/sklearn-crfsuite (tutorial:
https://sklearn-crfsuite.readthedocs.io/en/latest/tutorial.html)

[4] E. F. Tjong Kim Sang and F. De Meulder, "Introduction to the CoNLL-2003 shared task:
Language-independent named entity recognition," CoNLL 2003, pp. 142-147. arXiv:cs/0306050

[5] N. Ayoobi, S. Shahriar, and A. Mukherjee, "Seeing through AI's lens: Enhancing human skepticism
towards LLM-generated fake news," in Proc. 35th ACM Conf. Hypertext and Social Media (HT), 2024.
doi:10.1145/3648188.3675136

[6] D. Ashok and J. May, "A little human data goes a long way," arXiv:2410.13098, 2024.

[7] P. Koehn, "Statistical significance tests for machine translation evaluation," in Proc. EMNLP 2004,
pp. 388-395. https://aclanthology.org/W04-3250

[8] B. Efron and R. J. Tibshirani, An Introduction to the Bootstrap. Chapman & Hall/CRC, 1993.

[9] H. Nakayama, "seqeval: A Python framework for sequence labeling evaluation," 2018.
https://github.com/chakki-works/seqeval
